### Import

In [23]:
import os; import pandas as pd
pd.options.display.float_format = '{:.3f}'.format
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
import numpy as np; import matplotlib.pyplot as plt
import gurobipy as gp; from gurobipy import GRB
from itertools import product; from tqdm import tqdm
import importlib
import functions_utils; import functions_data
import functions_optimize; import functions_eval
importlib.reload(functions_data); importlib.reload(functions_optimize)
importlib.reload(functions_eval); importlib.reload(functions_utils)
from functions_utils import *; from functions_data import *
from functions_optimize import *; from functions_eval import *
import time

S = 20
LEVEL = "high"
SEED = 1

generation_data, I, T = load_generation_data(date_filter="2022-07-18")
R, P_RT, K, K0, M1, M2 = load_parameters(I, T, generation_data, S, LEVEL, SEED)
P_DA, P_PN = load_price_data(P_RT)

BASE_PATH = "/Users/jangseohyun/SynologyDrive/workspace/symply/DER/opt_result"


✅ 총 5개 파일을 불러왔습니다: 1201.csv, 137.csv, 401.csv, 524.csv, 89.csv
📊 데이터 Shape: I=5, T=24, S=20
✅ 시뮬레이션 초기화 완료: S=20, Randomness='high', Random Seed=1, M1=757.00, M2=2111.00


In [24]:
print("[Individual Participation Model optimization]")
x_ind, yp_ind, ym_ind, z_ind, zc_ind, zd_ind, OBJ_IND = optimize_individually_forall(R, K, K0, P_DA, P_RT, P_PN, I, T, S, M1)
print("-"*100)

print("[Holistic Aggregation Model optimization]")
x_hol, a_hol, yp_hol, ym_hol, z_hol, zc_hol, zd_hol, ep_hol, bp_hol, em_hol, bm_hol, d_hol, dp_hol, dm_hol, OBJ_HOL = optimize_hol(R, K, K0, P_DA, P_RT, P_PN, I, T, S, M1, M2)
print("-"*100)

[Individual Participation Model optimization]


Optimizing individually for each target_i:   0%|          | 0/5 [00:00<?, ?it/s]

Set parameter MIPGap to value 1e-07


Optimizing individually for each target_i:  20%|██        | 1/5 [00:00<00:00,  6.08it/s]

Optimal solution found for target_i=0! Objective value: 231671.1523130751
Set parameter MIPGap to value 1e-07


...

Optimizing individually for each target_i:  80%|████████  | 4/5 [00:00<00:00,  6.33it/s]

Optimal solution found for target_i=3! Objective value: 499211.63446583005
Set parameter MIPGap to value 1e-07


Optimizing individually for each target_i: 100%|██████████| 5/5 [00:00<00:00,  6.45it/s]

Optimal solution found for target_i=4! Objective value: 173152.49319508445
----------------------------------------------------------------------------------------------------
[Holistic Aggregation Model optimization]
Set parameter MIPGap to value 1e-06


Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (win64 - Windows 11.0 (26100.2))

CPU model: 13th Gen Intel(R) Core(TM) i9-13900K, instruction set [SSE2|AVX|AVX2]
Thread count: 24 physical cores, 32 logical processors, using up to 32 threads

Non-default parameters:
MIPGap  1e-06

Optimize a model with 36100 rows, 38620 columns and 105700 nonzeros
Model fingerprint: 0x95f77a9c
Variable types: 29020 continuous, 9600 integer (9600 binary)
Coefficient statistics:
  Matrix range     [1e+00, 8e+02]
  Objective range  [2e+00, 2e+02]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 8e+02]
Presolve removed 16489 rows and 19736 columns
Presolve time: 0.11s
Presolved: 19611 rows, 18884 columns, 59686 nonzeros
Variable types: 14409 continuous, 4475 integer (4475 binary)
Found heuristic solution: objective 1535962.5448
Found heuristic solution: objective 1536401.6401

Root relaxation: objective 1.807540e+06, 17964 iterations, 0.19 seconds (0.23 work units)

    Nodes    |    Current N

In [39]:
# internal pool price bounds

# Risk Neutral
MIN_BOUND = {t: np.mean([P_RT[t, s] for s in range(S)])-50 for t in range(T)}
MAX_BOUND = {t: np.mean([P_PN[t, s] for s in range(S)])+50 for t in range(T)}

# # Risk Averse
# MIN_BOUND = {t: np.max([P_RT[t, s] for s in range(S)]) for t in range(T)}
# MAX_BOUND = {t: np.min([P_PN[t, s] for s in range(S)]) for t in range(T)}
# for t in range(T):
#     if MIN_BOUND[t] >= MAX_BOUND[t]:
#         avg_value = (MIN_BOUND[t] + MAX_BOUND[t]) / 2
#         MIN_BOUND[t] = avg_value
#         MAX_BOUND[t] = avg_value

# Risk Seeking
# MIN_BOUND = {t: np.min([P_RT[t, s] for s in range(S)]) for t in range(T)}
# MAX_BOUND = {t: np.max([P_PN[t, s] for s in range(S)]) for t in range(T)}

### Endogeneous

In [53]:
def optimize_endogenous(R, K, K0, P_DA, P_RT, P_PN, I, T, S, M1, M2):
    
    model = gp.Model("DER_Aggregation_Endogenous")
    model.setParam("Heuristics", 0.3)
    model.setParam("MIPGap", 1e-4)
    model.setParam("TimeLimit", 60*30)
    
    # Decision Variables
    x = model.addVars(I, T, vtype=GRB.CONTINUOUS, lb=0, name="x")
    yp = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="ep") 
    ym = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="em")  
    dp = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="dp")  
    dm = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="dm")  
    z = model.addVars(I, T + 1, S, vtype=GRB.CONTINUOUS, lb=0, name="z")
    zc = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="zc") 
    zd = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="zd")
    phi1 = model.addVars(I, T, S, vtype=GRB.BINARY, name="phi1")
    phi2 = model.addVars(I, T, S, vtype=GRB.BINARY, name="phi2")
    phi3 = model.addVars(I, T, S, vtype=GRB.BINARY, name="phi3")
    phi4 = model.addVars(I, T, S, vtype=GRB.BINARY, name="phi4")
    phi5 = model.addVars(I, T, S, vtype=GRB.BINARY, name="phi5")
    phi6 = model.addVars(I, T, S, vtype=GRB.BINARY, name="phi6")
    phi7 = model.addVars(I, T, S, vtype=GRB.BINARY, name="phi7")
    # phi8 = model.addVars(T, S, vtype=GRB.BINARY, name="phi8")
    rhop = model.addVars(T, vtype=GRB.CONTINUOUS, name="rhop")
    rhom = model.addVars(T, vtype=GRB.CONTINUOUS, name="rhom")
    
    model.update()
    
    # Objective Function
    obj = (
        # Day-ahead revenue
        gp.quicksum(P_DA[t] * x[i, t] for i in range(I) for t in range(T)) +
        # Expected real-time revenue and penalty costs
        gp.quicksum((1/S) * (
            P_RT[t, s] * yp[i, t, s] - 
            P_PN[t, s] * ym[i, t, s] +
            rhop[t] * dp[i, t, s] -
            rhom[t] * dm[i, t, s]
        ) for i in range(I) for t in range(T) for s in range(S))
    )
    
    model.setObjective(obj, GRB.MAXIMIZE)
    
    # Constraints
    for i, t, s in product(range(I), range(T), range(S)):
        model.addConstr(R[i, t, s] - x[i, t] == yp[i, t, s] - ym[i, t, s] + dp[i, t, s] - dm[i, t, s] + zc[i, t, s] - zd[i, t, s])
        model.addConstr(R[i, t, s] + zd[i, t, s] >= yp[i, t, s] + dp[i, t, s] + zc[i, t, s])
        model.addConstr(zd[i, t, s] <= z[i, t, s])
        model.addConstr(zc[i, t, s] <= K[i] - z[i, t, s])
        model.addConstr(z[i, t, s] <= K[i])
        model.addConstr(z[i, t + 1, s] == z[i, t, s] + zc[i, t, s] - zd[i, t, s])
        
        # Logical constraints
        model.addConstr(yp[i, t, s] <= M1 * phi1[i, t, s]) ; model.addConstr(ym[i, t, s] <= M1 * (1 - phi1[i, t, s]))
        model.addConstr(ym[i, t, s] <= M1 * phi2[i, t, s]) ; model.addConstr(zc[i, t, s] <= M1 * (1 - phi2[i, t, s]))
        model.addConstr(zc[i, t, s] <= M1 * phi3[i, t, s]) ; model.addConstr(zd[i, t, s] <= M1 * (1 - phi3[i, t, s]))
        model.addConstr(dp[i, t, s] <= M1 * phi4[i, t, s]) ; model.addConstr(dm[i, t, s] <= M1 * (1 - phi4[i, t, s]))
        model.addConstr(zc[i, t, s] <= M1 * phi5[i, t, s]) ; model.addConstr(dm[i, t, s] <= M1 * (1 - phi5[i, t, s]))
        model.addConstr(yp[i, t, s] <= M1 * phi6[i, t, s]) ; model.addConstr(dm[i, t, s] <=  M1 * (1 - phi6[i, t, s]))
        model.addConstr(ym[i, t, s] <= M1 * phi7[i, t, s]) ; model.addConstr(dp[i, t, s] <= M1 * (1 - phi7[i, t, s]))

    for i, s in product(range(I), range(S)):
        model.addConstr(z[i, 0, s] == K0[i])

    for t, s in product(range(T), range(S)):
        model.addConstr(gp.quicksum(dp[i, t, s] for i in range(I)) == gp.quicksum(dm[i, t, s] for i in range(I)))
        # model.addConstr(gp.quicksum(yp[i, t, s] for i in range(I)) <= M2 * phi8[t, s])
        # model.addConstr(gp.quicksum(ym[i, t, s] for i in range(I)) <= M2 * (1 - phi8[t, s]))
    
    for t in range(T):
        model.addConstr(rhop[t] == rhom[t])
        # model.addConstr(rhop[t] <= 5000) # 물리적인 제약
        model.addConstr(rhop[t] >= MIN_BOUND[t]) # internal pool에서 파는 가격은 RT보다 커야함
        model.addConstr(rhom[t] <= MAX_BOUND[t]) # internal pool에서 사는 가격은 PN보다 작아야함
        # model.addConstr(rhom[t] >= P_DA[t])
    
    # model.addConstr(
    #     gp.quicksum(P_DA[t] * x[i, t] for i in range(I) for t in range(T)) +
    #     gp.quicksum((1/S) * (
    #         P_RT[t, s] * yp[i, t, s] - P_PN[t, s] * ym[i, t, s] 
    #         + rhop[t] * dp[i, t, s] - rhom[t] * dm[i, t, s]
    #     ) for i in range(I) for t in range(T) for s in range(S)) <= OBJ_HOL
    # )

    # for i in range(I):
    #     model.addConstr(
    #         gp.quicksum(P_DA[t] * x[i, t] for t in range(T)) +
    #         gp.quicksum((1/S) * (
    #             P_RT[t, s] * yp[i, t, s] - P_PN[t, s] * ym[i, t, s] +
    #             rhop[t] * dp[i, t, s] - rhom[t] * dm[i, t, s]
    #         ) for t in range(T) for s in range(S)) >= OBJ_IND[i]
    #     )

    # Optimize
    model.optimize()
    
    if model.status == GRB.OPTIMAL:
        print(f"Optimal solution found! Objective value: {model.objVal}")
    else:
        print("No optimal solution found.")
    
    # Extract solution
    x_sol = np.array([[x[i, t].X for t in range(T)] for i in range(I)])
    yp_sol = np.array([[[yp[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    ym_sol = np.array([[[ym[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    dp_sol = np.array([[[dp[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    dm_sol = np.array([[[dm[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    z_sol = np.array([[[z[i, t, s].X for s in range(S)] for t in range(T+1)] for i in range(I)])
    zc_sol = np.array([[[zc[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    zd_sol = np.array([[[zd[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    rhop_sol = np.array([rhop[t].X for t in range(T)])
    rhom_sol = np.array([rhom[t].X for t in range(T)])

    # Aggregate values
    a_sol = np.sum(x_sol, axis=0)  # Aggregate day-ahead commitment
    bp_sol = np.sum(yp_sol, axis=0)  # Aggregate real-time selling
    bm_sol = np.sum(ym_sol, axis=0)  # Aggregate real-time penalty
    dp_agg = np.sum(dp_sol, axis=0)  # Aggregate internal pool selling
    dm_agg = np.sum(dm_sol, axis=0)  # Aggregate internal pool buying
    
    return (x_sol, a_sol, yp_sol, ym_sol, z_sol, zc_sol, zd_sol, 
            dp_sol, dm_sol, bp_sol, bm_sol, dp_agg, dm_agg, model.objVal, rhop_sol, rhom_sol)

In [ ]:
x, a, yp, ym, z, zc, zd, dp, dm, bp, bm, dp_agg, dm_agg, obj_val, rhop, rhom = optimize_endogenous(R, K, K0, P_DA, P_RT, P_PN, I, T, S, M1, M2)

Set parameter Heuristics to value 0.3
Set parameter MIPGap to value 0.0001
Set parameter TimeLimit to value 1800
Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (win64 - Windows 11.0 (26100.2))

CPU model: 13th Gen Intel(R) Core(TM) i9-13900K, instruction set [SSE2|AVX|AVX2]
Thread count: 24 physical cores, 32 logical processors, using up to 32 threads

Non-default parameters:
TimeLimit  1800
Heuristics  0.3

Optimize a model with 48652 rows, 33868 columns and 120196 nonzeros
Model fingerprint: 0x47337282
Model has 4800 quadratic objective terms
Variable types: 17068 continuous, 16800 integer (16800 binary)
Coefficient statistics:
  Matrix range     [1e+00, 8e+02]
  Objective range  [2e+00, 2e+02]
  QObjective range [1e-01, 1e-01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 8e+02]
Presolve removed 18868 rows and 11397 columns
Presolve time: 0.25s
Presolved: 36981 rows, 26071 columns, 98486 nonzeros
Presolved model has 3598 bilinear constraint(s)

Solving non-convex MI

KeyboardInterrupt: 

Exception ignored in: 'gurobipy._core.logcallbackstub'
Traceback (most recent call last):
  File "c:\Users\jangseohyun\anaconda3\Lib\site-packages\ipykernel\iostream.py", line 655, in write
    def write(self, string: str) -> Optional[int]:  # type:ignore[override]
KeyboardInterrupt: 


H 3295  3442                    1797655.1298 2707661.56  50.6%   378  107s
  3442  3618 2698114.69   96 4979 1797655.13 2707661.56  50.6%   373  110s
  3816  3978 2697129.75  105 4906 1797655.13 2707661.56  50.6%   363  115s
H 3833  3978                    1797716.6242 2707661.56  50.6%   363  115s


In [42]:
# # 한 시나리오에 대해서 한명의 해 
# i = 3
# scen = 11

# header = (
#     f"{'s':>2} {'t':>2} | "
#     f"{'R':>8} {'x':>8} {'y+':>8} {'y-':>8} "
#     f"{'d+':>8} {'d-':>8} {'zc':>8} {'zd':>8} {'z':>8}\n"
#     + "-" * 90
# )
# # print(header)

# for s, t in product(range(scen,scen+1), range(8,22)):
#     print(header)
#     # individual
#     # print(
#     #     f"{s:>2} {t:>2} | "
#     #     f"{R[i, t, s]:>8.2f} {x_ind[i][t]:>8.2f} {yp_ind[i][t, s]:>8.2f} {ym_ind[i][t, s]:>8.2f} "
#     #     f"{0:>8.2f} {0:>8.2f} {zc_ind[i][t, s]:>8.2f} {zd_ind[i][t, s]:>8.2f} {z_ind[i][t, s]:>8.2f}"
#     #     )
#     # endogenous
#     print(
#         f"{s:>2} {t:>2} | "
#         f"{R[i, t, s]:>8.2f} {x[i, t]:>8.2f} {yp[i, t, s]:>8.2f} {ym[i, t, s]:>8.2f} "
#         f"{dp[i, t, s]:>8.2f} {dm[i, t, s]:>8.2f} {zc[i, t, s]:>8.2f} {zd[i, t, s]:>8.2f} {z[i, t, s]:>8.2f}"
#     )
#     # holistic
#     print(
#         f"{s:>2} {t:>2} | "
#         f"{R[i, t, s]:>8.2f} {x_hol[i, t]:>8.2f} {ep_hol[i, t, s]:>8.2f} {em_hol[i, t, s]:>8.2f} "
#         f"{dp_hol[i, t, s]:>8.2f} {dm_hol[i, t, s]:>8.2f} {zc_hol[i, t, s]:>8.2f} {zd_hol[i, t, s]:>8.2f} {z_hol[i, t, s]:>8.2f}"
#         )
#     print()

In [43]:
# # 한 시나리오에 대해서 전체합 
# scen = 0

# header = (
#     f"{'s':>2} {'t':>2} | "
#     f"{'R':>8} {'x':>8} {'y+':>8} {'y-':>8} "
#     f"{'d+':>8} {'d-':>8} {'zc':>8} {'zd':>8} {'z':>8}\n"
#     + "-" * 90
# )
# print(header)

# for s, t in product(range(scen, scen+1), range(8, 22)):
#     print(
#         f"{s:>2} {t:>2} | "
#         f"{R[:, t, s].sum():>8.2f} {x[:, t].sum():>8.2f} {yp[:, t, s].sum():>8.2f} {ym[:, t, s].sum():>8.2f} "
#         f"{dp[:, t, s].sum():>8.2f} {dm[:, t, s].sum():>8.2f} {zc[:, t, s].sum():>8.2f} {zd[:, t, s].sum():>8.2f} {z[:, t, s].sum():>8.2f}"
#     )
#     # print(
#     #         f"{s:>2} {t:>2} | "
#     #         f"{R[:, t, s].sum():>8.2f} {x_hol[:, t].sum():>8.2f} {yp_hol[:, t, s].sum():>8.2f} {ym_hol[:, t, s].sum():>8.2f} "
#     #         f"{dp_hol[:, t, s].sum():>8.2f} {dm_hol[:, t, s].sum():>8.2f} {zc_hol[:, t, s].sum():>8.2f} {zd_hol[:, t, s].sum():>8.2f} {z_hol[:, t, s].sum():>8.2f}"
#     # )

In [44]:
# 시나리오 평균으로 출력
header = (
    f"{'t':>2} | "
    f"{'R':>8} {'x':>8} {'y+':>8} {'y-':>8} "
    f"{'d+':>8} {'d-':>8} {'zc':>8} {'zd':>8} {'z':>8}\n"
    + "-" * 90
)
print(header)

for t in range(8, 22):
    # 각 변수의 시나리오 평균 계산
    R_avg = np.mean([R[:, t, s].sum() for s in range(S)])
    x_sum = x[:, t].sum()
    yp_avg = np.mean([yp[:, t, s].sum() for s in range(S)])
    ym_avg = np.mean([ym[:, t, s].sum() for s in range(S)])
    dp_avg = np.mean([dp[:, t, s].sum() for s in range(S)])
    dm_avg = np.mean([dm[:, t, s].sum() for s in range(S)])
    zc_avg = np.mean([zc[:, t, s].sum() for s in range(S)])
    zd_avg = np.mean([zd[:, t, s].sum() for s in range(S)])
    z_avg = np.mean([z[:, t, s].sum() for s in range(S)])
    
    print(
        f"{t:>2} | "
        f"{R_avg:>8.2f} {x_sum:>8.2f} {yp_avg:>8.2f} {ym_avg:>8.2f} "
        f"{dp_avg:>8.2f} {dm_avg:>8.2f} {zc_avg:>8.2f} {zd_avg:>8.2f} {z_avg:>8.2f}"
    )

 t |        R        x       y+       y-       d+       d-       zc       zd        z
------------------------------------------------------------------------------------------
 8 |    41.20     0.00     4.85     0.00     0.00     0.00    37.20     0.85    14.50
 9 |   184.70    18.00    35.05     0.00     0.00     0.00   136.55     4.90    50.85
10 |   440.05   166.00   145.65     0.00    28.55    28.55   167.10    38.70   182.50
11 |   655.15     0.00   535.70     0.00     0.00     0.00   174.30    54.85   310.90
12 |   885.20     0.00   816.35     0.00     0.00     0.00    68.85     0.00   430.35
13 |  1324.65     0.00  1728.15     0.00     0.00     0.00     0.00   403.50   499.20
14 |  1425.80   944.00   158.50    61.00   509.15   509.15   384.30     0.00    95.70
15 |   842.85     0.00  1322.85     0.00     0.00     0.00     0.00   480.00   480.00
16 |   763.95   441.00    83.50     0.00   275.10   275.10   239.45     0.00     0.00
17 |   865.45     0.00  1064.90     0.00     0.00

In [45]:
# 만약 hol 변수들도 출력하고 싶다면 아래 주석 해제
print("\n=== HOL Variables (평균) ===")
header_hol = (
    f"{'t':>2} | "
    f"{'R':>8} {'x_hol':>8} {'y+_hol':>8} {'y-_hol':>8} "
    f"{'d+_hol':>8} {'d-_hol':>8} {'zc_hol':>8} {'zd_hol':>8} {'z_hol':>8}\n"
    + "-" * 95
)
print(header_hol)

for t in range(8, 22):
    # HOL 변수들의 시나리오 평균 계산
    R_avg = np.mean([R[:, t, s].sum() for s in range(S)])
    x_hol_sum = x_hol[:, t].sum()  # x_hol은 시나리오와 무관
    ep_hol_avg = np.mean([ep_hol[:, t, s].sum() for s in range(S)])
    em_hol_avg = np.mean([em_hol[:, t, s].sum() for s in range(S)])
    dp_hol_avg = np.mean([dp_hol[:, t, s].sum() for s in range(S)])
    dm_hol_avg = np.mean([dm_hol[:, t, s].sum() for s in range(S)])
    zc_hol_avg = np.mean([zc_hol[:, t, s].sum() for s in range(S)])
    zd_hol_avg = np.mean([zd_hol[:, t, s].sum() for s in range(S)])
    z_hol_avg = np.mean([z_hol[:, t, s].sum() for s in range(S)])
    
    print(
        f"{t:>2} | "
        f"{R_avg:>8.2f} {x_hol_sum:>8.2f} {ep_hol_avg:>8.2f} {em_hol_avg:>8.2f} "
        f"{dp_hol_avg:>8.2f} {dm_hol_avg:>8.2f} {zc_hol_avg:>8.2f} {zd_hol_avg:>8.2f} {z_hol_avg:>8.2f}"
    )


=== HOL Variables (평균) ===
 t |        R    x_hol   y+_hol   y-_hol   d+_hol   d-_hol   zc_hol   zd_hol    z_hol
-----------------------------------------------------------------------------------------------
 8 |    41.20     0.00     4.85     0.00     0.00     0.00    37.20     0.85    14.50
 9 |   184.70    18.00    35.05     0.00     0.00     0.00   136.55     4.90    50.85
10 |   440.05   166.00   145.65     0.00    27.45    27.45   168.20    39.80   182.50
11 |   655.15     0.00   535.70     0.00     0.00     0.00   173.15    53.70   310.90
12 |   885.20     0.00   816.35     0.00     0.00     0.00    68.85     0.00   430.35
13 |  1324.65     0.00  1728.75     0.00     0.00     0.00     0.00   404.10   499.20
14 |  1425.80   926.00   172.00    57.10   489.05   489.05   384.90     0.00    95.10
15 |   842.85     0.00  1322.85     0.00     0.00     0.00     0.00   480.00   480.00
16 |   763.95   441.00    83.50     0.00   302.65   302.65   239.45     0.00     0.00
17 |   865.45   

In [46]:
for t in range(7, 21):
    dp_avg = np.mean([dp[:, t, s].sum() for s in range(S)]); dm_avg = np.mean([dm[:, t, s].sum() for s in range(S)])

    print(t, 'MIN', round(MIN_BOUND[t], 3), 'rhop', round(rhop[t], 3), 'dp', round(dp_avg, 3))
    print(t, 'MAX', round(MAX_BOUND[t], 3), 'rhom', round(rhom[t], 3), 'dm', round(dm_avg, 3))
    print()

7 MIN 33.818 rhop 197.399 dp 0.0
7 MAX 197.399 rhom 197.399 dm 0.0

8 MIN 52.227 rhop 137.87 dp 0.0
8 MAX 223.513 rhom 137.87 dm 0.0

9 MIN 51.213 rhop 143.035 dp 0.0
9 MAX 234.856 rhom 143.035 dm 0.0

10 MIN 68.871 rhop 160.395 dp 28.55
10 MAX 249.542 rhom 160.395 dm 28.55

11 MIN 89.982 rhop 179.712 dp 0.0
11 MAX 268.847 rhom 179.712 dm 0.0

12 MIN 133.177 rhop 230.631 dp 0.0
12 MAX 327.895 rhom 230.631 dm 0.0

13 MIN 221.327 rhop 339.159 dp 0.0
13 MAX 456.99 rhom 339.159 dm 0.0

14 MIN 77.197 rhop 196.855 dp 509.15
14 MAX 287.143 rhom 196.855 dm 509.15

15 MIN 254.293 rhop 380.367 dp 0.0
15 MAX 506.44 rhom 380.367 dm 0.0

16 MIN 96.254 rhop 207.654 dp 275.1
16 MAX 317.496 rhom 207.654 dm 275.1

17 MIN 168.94 rhop 275.388 dp 0.0
17 MAX 381.553 rhom 275.388 dm 0.0

18 MIN 106.143 rhop 203.543 dp 0.0
18 MAX 295.452 rhom 203.543 dm 0.0

19 MIN 51.682 rhop 159.966 dp 45.1
19 MAX 245.077 rhom 159.966 dm 45.1

20 MIN 48.874 rhop 155.103 dp 14.3
20 MAX 239.501 rhom 155.103 dm 14.3



In [47]:
profit = np.zeros(I)
for i in range(I):
    profit[i] = (
        # Day-ahead revenue
        sum(P_DA[t] * x[i, t] for t in range(T)) +
        # Expected real-time revenue and penalty costs
        sum((1/S) * (
            P_RT[t, s] * yp[i, t, s] - 
            P_PN[t, s] * ym[i, t, s] +
            rhop[t] * dp[i, t, s] -
            rhom[t] * dm[i, t, s]
        ) for t in range(T) for s in range(S))
    )

sum_=0
for i in range(I):
    print('ind', i, round(OBJ_IND[i],3))
    sum_ += OBJ_IND[i]
    print('model', i, round(profit[i],3))
    print()

print('OBJ_IND', round(sum_, 3))
print('OBJ_MODEL', round(obj_val, 3))
print('OBJ_HOL', round(OBJ_HOL,3))
print("------------------------")
print('verify', round(profit[:].sum(),3))


ind 0 231671.152
model 0 251949.054

ind 1 357308.065
model 1 393373.065

ind 2 400978.043
model 2 401492.919

ind 3 499211.634
model 3 547913.853

ind 4 173152.493
model 4 204420.621

OBJ_IND 1662321.388
OBJ_MODEL 1799149.511
OBJ_HOL 1799123.325
------------------------
verify 1799149.511


In [48]:
def compare_models(x, ep, em, dp, dm, rhop, rhom, obj_val,
                   x_hol, ep_hol, em_hol, dp_hol, dm_hol, OBJ_HOL,
                   P_DA, P_RT, P_PN, I, T, S):
    """
    Compare detailed profits between endogenous and holistic models
    """
    
    print("=" * 60)
    print("MODEL COMPARISON ANALYSIS")
    print("=" * 60)
    
    # 1. Overall objective value comparison
    print(f"\n1. OBJECTIVE VALUE COMPARISON")
    print(f"   Individual Model: {sum_:>12.2f}")
    print(f"   Endogenous Model: {obj_val:>12.2f}")
    print(f"   Holistic Model:   {OBJ_HOL:>12.2f}")
    
    # 2. Internal pool market clearing check
    print(f"\n2. INTERNAL POOL MARKET CLEARING CHECK")
    dp_sum = np.sum(dp, axis=0)  # (T, S) shape으로 만들기 - i에 대한 sum
    dm_sum = np.sum(dm, axis=0)  # (T, S) shape으로 만들기 - i에 대한 sum

    # Daily total check
    total_sell = np.sum(rhop.reshape(-1, 1) * dp_sum)  # (T, 1) * (T, S)
    total_buy = np.sum(rhom.reshape(-1, 1) * dm_sum)   # (T, 1) * (T, S)
    print(f"   Daily Total:")
    print(f"     Total Sell Value: {total_sell:>12.2f}")
    print(f"     Total Buy Value:  {total_buy:>12.2f}")
    print(f"     Net Transfer:     {total_sell - total_buy:>12.2f}")
    print(f"     Balanced: {'✓' if abs(total_sell - total_buy) < 1e-6 else '✗'}")

    # Hourly check (show violations if any)
    print(f"\n   Hourly Check:")
    violations = []
    for t in range(T):
        hourly_sell = np.sum(rhop[t] * dp_sum[t, :])  # 시간 t에서 모든 시나리오 합
        hourly_buy = np.sum(rhom[t] * dm_sum[t, :])   # 시간 t에서 모든 시나리오 합
        net_transfer = hourly_sell - hourly_buy
        
        if abs(net_transfer) > 1e-6:
            violations.append((t, hourly_sell, hourly_buy, net_transfer))

    if violations:
        print(f"     Found {len(violations)} hourly violations:")
        print(f"     {'Hour':>4} | {'Sell':>12} | {'Buy':>12} | {'Net':>12}")
        print(f"     {'-'*4} | {'-'*12} | {'-'*12} | {'-'*12}")
        for t, sell, buy, net in violations[:10]:  # Show first 10
            print(f"     {t:>4} | {sell:>12.2f} | {buy:>12.2f} | {net:>12.2f}")
        if len(violations) > 10:
            print(f"     ... and {len(violations) - 10} more")
    else:
        print(f"     All hourly balances satisfied ✓")

    # Market clearing quantity check (without prices)
    print(f"\n   Quantity Balance Check:")
    qty_violations = []
    for t in range(T):
        for s in range(S):
            sell_qty = dp_sum[t, s]
            buy_qty = dm_sum[t, s]
            if abs(sell_qty - buy_qty) > 1e-6:
                qty_violations.append((t, s, sell_qty, buy_qty))

    if qty_violations:
        print(f"     Found {len(qty_violations)} quantity violations:")
        print(f"     {'Hour':>4} | {'Scen':>4} | {'Sell Qty':>12} | {'Buy Qty':>12} | {'Diff':>12}")
        print(f"     {'-'*4} | {'-'*4} | {'-'*12} | {'-'*12} | {'-'*12}")
        for t, s, sell, buy in qty_violations[:10]:
            print(f"     {t:>4} | {s:>4} | {sell:>12.2f} | {buy:>12.2f} | {sell-buy:>12.2f}")
        if len(qty_violations) > 10:
            print(f"     ... and {len(qty_violations) - 10} more")
    else:
        print(f"     All quantity balances satisfied ✓")
        
    # 3. Day-ahead profit comparison
    print(f"\n3. DAY-AHEAD PROFIT COMPARISON")
    da_profit_endo = np.sum([P_DA[t] * np.sum(x[:, t]) for t in range(T)])
    da_profit_hol = np.sum([P_DA[t] * np.sum(x_hol[:, t]) for t in range(T)])
    print(f"   Endogenous Model: {da_profit_endo:>12.2f}")
    print(f"   Holistic Model:   {da_profit_hol:>12.2f}")
    print(f"   Difference:       {da_profit_endo - da_profit_hol:>12.2f}")
    
    # 4. Day-ahead commitment quantity comparison
    print(f"\n4. DAY-AHEAD COMMITMENT QUANTITY")
    da_qty_endo = np.sum(x)
    da_qty_hol = np.sum(x_hol)
    print(f"   Endogenous Model: {da_qty_endo:>12.2f}")
    print(f"   Holistic Model:   {da_qty_hol:>12.2f}")
    print(f"   Difference:       {da_qty_endo - da_qty_hol:>12.2f}")
    
    # 5. Real-time profit comparison
    print(f"\n5. REAL-TIME PROFIT COMPARISON")
    rt_profit_endo = np.sum([P_RT[t, s] * np.sum(ep[:, t, s]) for t in range(T) for s in range(S)]) / S
    rt_profit_hol = np.sum([P_RT[t, s] * np.sum(ep_hol[:, t, s]) for t in range(T) for s in range(S)]) / S
    print(f"   Endogenous Model: {rt_profit_endo:>12.2f}")
    print(f"   Holistic Model:   {rt_profit_hol:>12.2f}")
    print(f"   Difference:       {rt_profit_endo - rt_profit_hol:>12.2f}")
    
    # 6. Real-time quantity comparison
    print(f"\n6. REAL-TIME QUANTITY COMPARISON")
    rt_qty_endo = np.mean(np.sum(ep, axis=(0, 1)))
    rt_qty_hol = np.mean(np.sum(ep_hol, axis=(0, 1)))
    print(f"   Endogenous Model: {rt_qty_endo:>12.2f}")
    print(f"   Holistic Model:   {rt_qty_hol:>12.2f}")
    print(f"   Difference:       {rt_qty_endo - rt_qty_hol:>12.2f}")
    
    # 7. Penalty comparison
    print(f"\n7. PENALTY COMPARISON")
    penalty_endo = np.sum([P_PN[t, s] * np.sum(em[:, t, s]) for t in range(T) for s in range(S)]) / S
    penalty_hol = np.sum([P_PN[t, s] * np.sum(em_hol[:, t, s]) for t in range(T) for s in range(S)]) / S
    print(f"   Endogenous Model: {penalty_endo:>12.2f}")
    print(f"   Holistic Model:   {penalty_hol:>12.2f}")
    print(f"   Difference:       {penalty_endo - penalty_hol:>12.2f}")
    
    # 8. Internal pool trading comparison
    print(f"\n8. INTERNAL POOL TRADING")
    pool_trade_endo = np.mean(np.sum(dp + dm, axis=(0, 1)))
    pool_trade_hol = np.mean(np.sum(dp_hol + dm_hol, axis=(0, 1)))
    print(f"   Endogenous Model: {pool_trade_endo:>12.2f}")
    print(f"   Holistic Model:   {pool_trade_hol:>12.2f}")
    print(f"   Difference:       {pool_trade_endo - pool_trade_hol:>12.2f}")
    
    # 9. Individual DER comparison (first few DERs)
    print(f"\n9. INDIVIDUAL DER PROFIT COMPARISON")
    print(f"   {'DER':>3} | {'Individual':>12} | {'Endogenous':>12} | {'Holistic':>12} |")
    print(f"   {'-'*3} | {'-'*12} | {'-'*12} | {'-'*12} |")
    
    for i in range(I):
        profit_ind = OBJ_IND[i]
        profit_endo = (
            sum(P_DA[t] * x[i, t] for t in range(T)) +
            sum((1/S) * (P_RT[t, s] * ep[i, t, s] - P_PN[t, s] * em[i, t, s] +
            rhop[t] * dp[i, t, s] - rhom[t] * dm[i, t, s]) 
                for t in range(T) for s in range(S))
        )
        profit_hol = (
            sum(P_DA[t] * x_hol[i, t] for t in range(T)) +
            sum((1/S) * (P_RT[t, s] * ep_hol[i, t, s] - P_PN[t, s] * em_hol[i, t, s]) 
                for t in range(T) for s in range(S))
        )
        print(f"   {i:>3} | {profit_ind:>12.2f} | {profit_endo:>12.2f} | {profit_hol:>12.2f} |")
    
    print("=" * 60)

# 사용법
compare_models(x, yp, ym, dp, dm, rhop, rhom, obj_val,
               x_hol, ep_hol, em_hol, dp_hol, dm_hol, OBJ_HOL,
               P_DA, P_RT, P_PN, I, T, S)

MODEL COMPARISON ANALYSIS

1. OBJECTIVE VALUE COMPARISON
   Individual Model:   1662321.39
   Endogenous Model:   1799149.51
   Holistic Model:     1799123.32

2. INTERNAL POOL MARKET CLEARING CHECK
   Daily Total:
     Total Sell Value:   3430317.16
     Total Buy Value:    3430317.16
     Net Transfer:            -0.00
     Balanced: ✓

   Hourly Check:
     All hourly balances satisfied ✓

   Quantity Balance Check:
     All quantity balances satisfied ✓

3. DAY-AHEAD PROFIT COMPARISON
   Endogenous Model:    307314.01
   Holistic Model:      304470.21
   Difference:            2843.80

4. DAY-AHEAD COMMITMENT QUANTITY
   Endogenous Model:      2008.00
   Holistic Model:        1990.00
   Difference:              18.00

5. REAL-TIME PROFIT COMPARISON
   Endogenous Model:   1506291.50
   Holistic Model:     1508184.88
   Difference:           -1893.38

6. REAL-TIME QUANTITY COMPARISON
   Endogenous Model:      6556.85
   Holistic Model:        6570.95
   Difference:             -14.1

### Individual Replay

In [49]:
def individual_replay(R, K, K0, P_DA, P_RT, P_PN, RHOP, RHOM, I, T, S, M1):
    
    model = gp.Model("DER_Individual_Replay")
    model.setParam("Heuristics", 0.2)
    model.setParam("TimeLimit", 60*15)
    
    x = model.addVars(I, T, vtype=GRB.CONTINUOUS, lb=0, name="x")
    yp = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="yp") 
    ym = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="ym")  
    dp = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="dp")  
    dm = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="dm")  
    z = model.addVars(I, T + 1, S, vtype=GRB.CONTINUOUS, lb=0, name="z")
    zc = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="zc") 
    zd = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="zd")
    phi1 = model.addVars(I, T, S, vtype=GRB.BINARY, name="phi1")
    phi2 = model.addVars(I, T, S, vtype=GRB.BINARY, name="phi2")
    phi3 = model.addVars(I, T, S, vtype=GRB.BINARY, name="phi3")
    phi4 = model.addVars(I, T, S, vtype=GRB.BINARY, name="phi4")
    phi5 = model.addVars(I, T, S, vtype=GRB.BINARY, name="phi5")
    phi6 = model.addVars(I, T, S, vtype=GRB.BINARY, name="phi6")
    phi7 = model.addVars(I, T, S, vtype=GRB.BINARY, name="phi7")
    
    model.update()
    
    obj = (
        gp.quicksum(P_DA[t] * x[i, t] for i in range(I) for t in range(T)) +
        gp.quicksum((1/S) * (
            P_RT[t, s] * yp[i, t, s] - P_PN[t, s] * ym[i, t, s] +
            RHOP[i, t, s] * dp[i, t, s] - RHOM[i, t, s] * dm[i, t, s]
        ) for i in range(I) for t in range(T) for s in range(S))
    )
    
    model.setObjective(obj, GRB.MAXIMIZE)
    
    for i, t, s in product(range(I), range(T), range(S)):
        model.addConstr(R[i, t, s] - x[i, t] == yp[i, t, s] - ym[i, t, s] + dp[i, t, s] - dm[i, t, s] + zc[i, t, s] - zd[i, t, s])
        model.addConstr(R[i, t, s] + zd[i, t, s] >= yp[i, t, s] + dp[i, t, s] + zc[i, t, s])
        model.addConstr(zd[i, t, s] <= z[i, t, s])
        model.addConstr(zc[i, t, s] <= K[i] - z[i, t, s])
        model.addConstr(z[i, t, s] <= K[i])
        model.addConstr(z[i, t + 1, s] == z[i, t, s] + zc[i, t, s] - zd[i, t, s])
        
        model.addConstr(yp[i, t, s] <= M1 * phi1[i, t, s]) ; model.addConstr(ym[i, t, s] <= M1 * (1 - phi1[i, t, s]))
        model.addConstr(ym[i, t, s] <= M1 * phi2[i, t, s]) ; model.addConstr(zc[i, t, s] <= M1 * (1 - phi2[i, t, s]))
        model.addConstr(zc[i, t, s] <= M1 * phi3[i, t, s]) ; model.addConstr(zd[i, t, s] <= M1 * (1 - phi3[i, t, s]))
        model.addConstr(dp[i, t, s] <= M1 * phi4[i, t, s]) ; model.addConstr(dm[i, t, s] <= M1 * (1 - phi4[i, t, s]))
        model.addConstr(zc[i, t, s] <= M1 * phi5[i, t, s]) ; model.addConstr(dm[i, t, s] <= M1 * (1 - phi5[i, t, s]))
        model.addConstr(yp[i, t, s] <= M1 * phi6[i, t, s]) ; model.addConstr(dm[i, t, s] <= M1 * (1 - phi6[i, t, s]))
        model.addConstr(ym[i, t, s] <= M1 * phi7[i, t, s]) ; model.addConstr(dp[i, t, s] <= M1 * (1 - phi7[i, t, s]))

    for i, s in product(range(I), range(S)):
        model.addConstr(z[i, 0, s] == K0[i])

    model.optimize()
    
    if model.status == GRB.OPTIMAL:
        print(f"Optimal solution found! Objective value: {model.objVal}")
    else:
        print("No optimal solution found.")
    
    # Extract solution
    x_sol = np.array([[x[i, t].X for t in range(T)] for i in range(I)])
    yp_sol = np.array([[[yp[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    ym_sol = np.array([[[ym[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    dp_sol = np.array([[[dp[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    dm_sol = np.array([[[dm[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    z_sol = np.array([[[z[i, t, s].X for s in range(S)] for t in range(T+1)] for i in range(I)])
    zc_sol = np.array([[[zc[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    zd_sol = np.array([[[zd[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    
    return (x_sol, yp_sol, ym_sol, z_sol, zc_sol, zd_sol, dp_sol, dm_sol, model.objVal)

In [50]:
RHOP, RHOM = np.full((I,T,S), -1e10), np.full((I,T,S), 1e10)

# for i, t, s in product(range(I), range(T), range(S)):
#     RHOP[i, t, s] = rhop[t]
#     RHOM[i, t, s] = rhom[t]

for i, t, s in product(range(I), range(T), range(S)):
    if dp[i, t, s] > 0 and dm[i, t, s] == 0:
        RHOP[i, t, s] = rhop[t]
        RHOM[i, t, s] = 1e10
    elif dm[i, t, s] > 0 and dp[i, t, s] == 0:
        RHOP[i, t, s] = -1e10
        RHOM[i, t, s] = rhom[t] 

x_re, yp_re, ym_re, z_re, zc_re, zd_re, dp_re, dm_re, OBJ_RE = individual_replay(R, K, K0, P_DA, P_RT, P_PN, RHOP, RHOM, I, T, S, M1)

Set parameter Heuristics to value 0.2
Set parameter TimeLimit to value 900
Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (win64 - Windows 11.0 (26100.2))

CPU model: 13th Gen Intel(R) Core(TM) i9-13900K, instruction set [SSE2|AVX|AVX2]
Thread count: 24 physical cores, 32 logical processors, using up to 32 threads

Non-default parameters:
TimeLimit  900
Heuristics  0.2

Optimize a model with 48100 rows, 33820 columns and 115300 nonzeros
Model fingerprint: 0x604cb74a
Variable types: 17020 continuous, 16800 integer (16800 binary)
Coefficient statistics:
  Matrix range     [1e+00, 8e+02]
  Objective range  [2e+00, 5e+08]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 8e+02]
Found heuristic solution: objective -3.62800e+13
Presolve removed 41627 rows and 28895 columns
Presolve time: 0.29s
Presolved: 6473 rows, 4925 columns, 16151 nonzeros
Found heuristic solution: objective -8.57350e+12
Variable types: 2503 continuous, 2422 integer (2422 binary)

Root relaxation: objective 

In [51]:
# 시나리오 평균으로 출력
header = (
    f"{'t':>2} | "
    f"{'R':>8} {'x':>8} {'y+':>8} {'y-':>8} "
    f"{'d+':>8} {'d-':>8} {'zc':>8} {'zd':>8} {'z':>8}\n"
    + "-" * 90
)
print(header)

for t in range(8, 22):
    # 각 변수의 시나리오 평균 계산
    R_avg = np.mean([R[:, t, s].sum() for s in range(S)])
    x_sum = x_re[:, t].sum()
    yp_avg = np.mean([yp_re[:, t, s].sum() for s in range(S)])
    ym_avg = np.mean([ym_re[:, t, s].sum() for s in range(S)])
    dp_avg = np.mean([dp_re[:, t, s].sum() for s in range(S)])
    dm_avg = np.mean([dm_re[:, t, s].sum() for s in range(S)])
    zc_avg = np.mean([zc_re[:, t, s].sum() for s in range(S)])
    zd_avg = np.mean([zd_re[:, t, s].sum() for s in range(S)])
    z_avg = np.mean([z_re[:, t, s].sum() for s in range(S)])
    
    print(
        f"{t:>2} | "
        f"{R_avg:>8.2f} {x_sum:>8.2f} {yp_avg:>8.2f} {ym_avg:>8.2f} "
        f"{dp_avg:>8.2f} {dm_avg:>8.2f} {zc_avg:>8.2f} {zd_avg:>8.2f} {z_avg:>8.2f}"
    )

 t |        R        x       y+       y-       d+       d-       zc       zd        z
------------------------------------------------------------------------------------------
 8 |    41.20     0.00     4.60     0.00     0.00     0.00    37.05     0.45    14.45
 9 |   184.70    17.00    34.20     0.00     0.00     0.00   139.20     5.70    51.05
10 |   440.05    93.00   176.05     0.00    85.30     1.60   153.15    65.85   184.55
11 |   655.15     0.00   496.65     0.00     0.00     0.00   203.75    45.25   271.85
12 |   885.20     0.00   826.25     0.00     0.00     0.00    68.85     9.90   430.35
13 |  1324.65     0.00  1797.65     0.00     0.00     0.00     0.00   473.00   489.30
14 |  1425.80   126.00   247.20     0.00   568.90     0.00   483.70     0.00    16.30
15 |   842.85     0.00  1342.85     0.00     0.00     0.00     0.00   500.00   500.00
16 |   763.95     0.00    56.30     0.00   438.75     0.00   268.90     0.00     0.00
17 |   865.45     0.00  1092.45     0.00     0.00

In [52]:
# 한 명 시나리오 평균으로 출력
header = (
    f"{'t':>2} | "
    f"{'R':>8} {'x':>8} {'y+':>8} {'y-':>8} "
    f"{'d+':>8} {'d-':>8} {'zc':>8} {'zd':>8} {'z':>8}\n"
    + "-" * 90
)
print(header)
i=0
for t in range(8, 22):
    # 각 변수의 시나리오 평균 계산
    R_avg = np.mean([R[i, t, s] for s in range(S)])
    x_sum = x_re[i, t]
    yp_avg = np.mean([yp_re[i, t, s] for s in range(S)])
    ym_avg = np.mean([ym_re[i, t, s] for s in range(S)])
    dp_avg = np.mean([dp_re[i, t, s] for s in range(S)])
    dm_avg = np.mean([dm_re[i, t, s] for s in range(S)])
    zc_avg = np.mean([zc_re[i, t, s] for s in range(S)])
    zd_avg = np.mean([zd_re[i, t, s] for s in range(S)])
    z_avg = np.mean([z_re[i, t, s] for s in range(S)])
    
    print(
        f"{t:>2} | "
        f"{R_avg:>8.2f} {x_sum:>8.2f} {yp_avg:>8.2f} {ym_avg:>8.2f} "
        f"{dp_avg:>8.2f} {dm_avg:>8.2f} {zc_avg:>8.2f} {zd_avg:>8.2f} {z_avg:>8.2f}"
    )

 t |        R        x       y+       y-       d+       d-       zc       zd        z
------------------------------------------------------------------------------------------
 8 |    16.15     0.00     0.00     0.00     0.00     0.00    16.15     0.00     0.00
 9 |     3.65     0.00     0.00     0.00     0.00     0.00     3.65     0.00    16.15
10 |    39.65     0.00     7.75     0.00     1.55     0.00    30.40     0.05    19.80
11 |    68.20     0.00    29.80     0.00     0.00     0.00    40.55     2.15    50.15
12 |   155.90     0.00   144.45     0.00     0.00     0.00    11.45     0.00    88.55
13 |   224.90     0.00   321.15     0.00     0.00     0.00     0.00    96.25   100.00
14 |   212.05     0.00    16.95     0.00    98.85     0.00    96.25     0.00     3.75
15 |   168.30     0.00   268.30     0.00     0.00     0.00     0.00   100.00   100.00
16 |    18.95     0.00     1.25     0.00     0.00     0.00    17.70     0.00     0.00
17 |     0.00     0.00    15.60     0.00     0.00